In [2]:
import os
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from pathlib import Path
import json
import random
import logging
from datetime import datetime
import time

import openai
from langchain import OpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader
import numpy as np

In [3]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class QuestionAnswer:
    question: str
    answer: str
    category: str
    difficulty: str
    score: float = 0.0
    feedback: Dict = None

class InterviewSystem:
    def __init__(self, openai_api_key: str, pdf_path: str, vector_store_path: str = "vector_store"):
        """Initialize the interview system with necessary components.
        Args:
            openai_api_key: API key for OpenAI
            pdf_path: Path to the data science PDF book
            vector_store_path: Path to store/load vector embeddings
        """
        self.openai_api_key = openai_api_key
        self.pdf_path = Path(pdf_path)
        self.vector_store_path = Path(vector_store_path)
        
        # Initialize OpenAI and embedding models
        self.llm = OpenAI(openai_api_key=openai_api_key)
        self.embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
        
        # Initialize storage for QA pairs
        self.qa_pairs: List[QuestionAnswer] = []
        
        # Set up vector store
        self._setup_vector_store()

    def _setup_vector_store(self) -> None:
        """Set up the vector store for RAG implementation."""
        try:
            if self.vector_store_path.exists():
                self.vector_store = FAISS.load_local(
                    self.vector_store_path, self.embeddings, allow_dangerous_deserialization=True
                )
                logger.info("Loaded existing vector store")
            else:
                # Load and process PDF
                logger.info('Start processing PDF file')
                loader = PyPDFLoader(str(self.pdf_path))
                documents = loader.load()
                
                # Split documents into chunks
                logger.info('Start splitting PDF file into chunks')

                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=200
                )
                texts = text_splitter.split_documents(documents)
                
                # Create and save vector store
                logger.info('Start creating vector store')
                self.vector_store = FAISS.from_documents(texts, self.embeddings)
                logger.info('Finished loading vector embeddings')
                self.vector_store.save_local(self.vector_store_path)
                logger.info("Created and saved new vector store")
        
        except Exception as e:
            logger.error(f"Error setting up vector store: {str(e)}")
            raise

    def generate_qa_pairs(self, num_pairs: int = 50) -> List[QuestionAnswer]:
        """Generate question-answer pairs using RAG.
        
        Args:
            num_pairs: Number of QA pairs to generate
            
        Returns:
            List of QuestionAnswer objects
        """
        qa_generator_prompt = """
        Generate a data science interview question and comprehensive answer based on the provided context.
        The question should be challenging but clear, and the answer should demonstrate mastery of the concept.
        
        Format:
        {
            "question": "Question text",
            "answer": "Detailed answer",
            "category": "One of: ML, Statistics, Programming, Math, Domain Knowledge",
            "difficulty": "One of: Easy, Medium, Hard"
        }
        """
        
        retriever = self.vector_store.as_retriever(
            search_kwargs={"k": 3}
        )
        
        qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True
        )
        
        generated_pairs = []
        for _ in range(num_pairs):
            try:
                # Get random context from vector store
                docs = retriever.get_relevant_documents("")
                context = " ".join([doc.page_content for doc in docs])
                
                # Generate QA pair
                result = qa_chain({"query": qa_generator_prompt + "\n\nContext: " + context})
                qa_data = json.loads(result["result"])
                
                qa_pair = QuestionAnswer(**qa_data)
                generated_pairs.append(qa_pair)
                
                # Add delay to respect rate limits
                time.sleep(1)
                
            except Exception as e:
                logger.error(f"Error generating QA pair: {str(e)}")
                continue
        
        self.qa_pairs.extend(generated_pairs)
        return generated_pairs

    def score_question(self, qa_pair: QuestionAnswer) -> float:
        """Score the quality of a question-answer pair.
        
        Args:
            qa_pair: QuestionAnswer object to score
            
        Returns:
            Float score between 0 and 100
        """
        scoring_prompt = """
        Score the following data science interview question and answer on a scale of 0-100.
        Consider:
        1. Relevance (25 points): Does it test essential data science knowledge?
        2. Clarity (25 points): Is the question clear and unambiguous?
        3. Technical Depth (25 points): Does it require deep understanding?
        4. Practical Value (25 points): Is it applicable to real-world scenarios?

        Question: {question}
        Answer: {answer}
        Category: {category}
        Difficulty: {difficulty}

        Provide score and breakdown in JSON format:
        {{"total_score": float, "breakdown": {{"relevance": float, "clarity": float, "technical_depth": float, "practical_value": float}}, "feedback": string}}
        """
        
        try:

            from openai import OpenAI
            client = OpenAI()

            response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                    "role": "user",
                    "content": scoring_prompt.format(**qa_pair.__dict__)
                }],
            temperature=0.3
            )
            
            result = json.loads(response.choices[0].message.content)
            logger.info(f'Scoring Question Result: {result}')
            qa_pair.score = result["total_score"]
            qa_pair.feedback = result["breakdown"]
            
            return result["total_score"]
            
        except Exception as e:
            logger.error(f"Error scoring question: {str(e)}")
            return 0.0

    def select_interview_question(self) -> Optional[QuestionAnswer]:
        """Select a random question from the top 36 highest-scoring questions."""
        if not self.qa_pairs:
            logger.warning("No questions available to select from")
            return None
            
        # Sort by score and get top 36
        top_questions = sorted(
            self.qa_pairs, 
            key=lambda x: x.score, 
            reverse=True
        )[:36]
        
        if not top_questions:
            return random.choice(self.qa_pairs)
            
        return random.choice(top_questions)

    def evaluate_answer(self, question: QuestionAnswer, candidate_answer: str) -> Tuple[float, Dict]:
        """Evaluate a candidate's answer against the model answer.
        
        Args:
            question: QuestionAnswer object containing the question and model answer
            candidate_answer: The candidate's response
            
        Returns:
            Tuple of (score, feedback_dict)
        """
        evaluation_prompt = """
        Evaluate the candidate's answer against the model answer for this data science interview question.
        Score from 1-10 and provide detailed feedback.

        Question: {question}
        Model Answer: {model_answer}
        Candidate Answer: {candidate_answer}

        Provide evaluation in JSON format:
        {{"score": float, "feedback": {{"strengths": [str], "areas_for_improvement": [str], "missing_key_points": [str]}}}}
        """
        
        try:

            from openai import OpenAI
            client = OpenAI()

            response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                    "role": "user",
                    "content": evaluation_prompt.format(
                        question=question.question,
                        model_answer=question.answer,
                        candidate_answer=candidate_answer
                    )
                }],
            temperature=0.3
            )
            
            result = json.loads(response.choices[0].message.content)
            return result["score"], result["feedback"]
            
        except Exception as e:
            logger.error(f"Error evaluating answer: {str(e)}")
            return 0.0, {"error": str(e)}



In [4]:
# Example usage
from dotenv import load_dotenv

if __name__ == "__main__":
    # Load environment variables
    # openai_api_key = os.getenv("OPENAI_API_KEY")
    
    load_dotenv()
    openai_api_key = os.getenv('OPENAI_API_KEY')
    # Initialize system
    interview_system = InterviewSystem(
        openai_api_key=openai_api_key,
        pdf_path="inputs/contents/book.pdf"
    )
    
    # Generate QA pairs
    qa_pairs = interview_system.generate_qa_pairs(num_pairs=10)
    
    # Score questions
    for qa_pair in qa_pairs:
        score = interview_system.score_question(qa_pair)
        print(f"Question scored: {score}")
    
    # Select and ask a question
    selected_question = interview_system.select_interview_question()
    if selected_question:
        print("\nSelected Question:")
        print(f"Q: {selected_question.question}")
        
        # Simulate candidate answer
        candidate_answer = input("\nYour answer: ")
        
        # Evaluate answer
        score, feedback = interview_system.evaluate_answer(
            selected_question, 
            candidate_answer
        )
        
        print(f"\nScore: {score}/10")
        print("Feedback:", json.dumps(feedback, indent=2))

/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAI`.
  warn_deprecated(
/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(
INFO:faiss.loader:Loading faiss.
INFO:faiss.l

Question scored: 0.0


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:__main__:Error scoring question: Expecting value: line 1 column 1 (char 0)


Question scored: 0.0


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:__main__:Error scoring question: Expecting value: line 1 column 1 (char 0)


...

Question scored: 0.0


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
ERROR:__main__:Error scoring question: Expecting value: line 1 column 1 (char 0)


Question scored: 0.0

Selected Question:
Q: What is the argument for why clustering words of high probability is more important than clustering words of low probability?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Score: 1.0/10
Feedback: {
  "strengths": [],
  "areas_for_improvement": [
    "The candidate did not provide a relevant answer to the question.",
    "The candidate's response lacks depth and does not address the core argument of the question."
  ],
  "missing_key_points": [
    "The importance of high probability words in conveying meaning and context.",
    "The impact of low probability words on overall document understanding.",
    "A clear comparison between the significance of clustering high versus low probability words."
  ]
}


In [35]:
qa_pairs

[QuestionAnswer(question='What is the importance of clustering words by probability in document classification?', answer='Clustering words by probability is important in document classification because it allows us to prioritize the most significant words in a document. High probability words are more likely to be unique to a particular document, making them more informative for classification. On the other hand, low probability words are less likely to have a strong impact on classification since they are more common and may appear in multiple documents. By clustering words based on probability, we can focus on the most important words and reduce the impact of less significant words on our classification model.', category='ML', difficulty='Medium', score=0.0, feedback=None),
 QuestionAnswer(question='How does the concept of probability play a role in text clustering algorithms?', answer='Probability plays a crucial role in text clustering algorithms as it allows us to assign weights t

In [36]:
selected_question

QuestionAnswer(question='What is the importance of clustering words by probability in document classification?', answer='Clustering words by probability is important in document classification because it allows us to prioritize the most significant words in a document. High probability words are more likely to be unique to a particular document, making them more informative for classification. On the other hand, low probability words are less likely to have a strong impact on classification since they are more common and may appear in multiple documents. By clustering words based on probability, we can focus on the most important words and reduce the impact of less significant words on our classification model.', category='ML', difficulty='Medium', score=0.0, feedback=None)

In [ ]:
interview_system.score_question(qa_pair)

In [ ]:
qa_pair

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 15.0 kB/s eta 0:00:00a 0:00:02

[notice] A new release of pip is available: 24.1.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from pathlib import Path
import json
import random
import logging
from datetime import datetime
import time

import anthropic
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader
import numpy as np

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class QuestionAnswer:
    """Data class for storing question-answer pairs with metadata."""
    question: str
    answer: str
    category: str
    difficulty: str
    score: float = 0.0
    feedback: Dict = None
    source: str = None

class InterviewSystem:
    def __init__(self, anthropic_api_key: str, pdf_paths: List[str], vector_store_path: str = "vector_store"):
        """Initialize the interview system with necessary components.
        
        Args:
            anthropic_api_key: API key for Anthropic Claude
            pdf_paths: List of paths to data science PDF books
            vector_store_path: Path to store/load vector embeddings
        """
        self.client = anthropic.Client(api_key=anthropic_api_key)
        self.pdf_paths = [Path(path) for path in pdf_paths]
        self.vector_store_path = Path(vector_store_path)
        
        # Validate PDF paths
        for pdf_path in self.pdf_paths:
            if not pdf_path.exists():
                raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        
        # Initialize storage for QA pairs
        self.qa_pairs: List[QuestionAnswer] = []
        
        # Set up vector store
        self._setup_vector_store()

    def _setup_vector_store(self) -> None:
        """Set up the vector store for RAG implementation."""
        try:
            if self.vector_store_path.exists():
                self.vector_store = FAISS.load_local(
                    self.vector_store_path, 
                    self.embeddings,
                    allow_dangerous_deserialization=True
                )
                logger.info("Loaded existing vector store")
            else:
                # Load and process all PDFs
                documents = []
                for pdf_path in self.pdf_paths:
                    try:
                        logger.info(f'Processing PDF: {pdf_path}')
                        loader = PyPDFLoader(str(pdf_path))
                        pdf_documents = loader.load()
                        
                        # Add source metadata to each document
                        for doc in pdf_documents:
                            doc.metadata['source_pdf'] = str(pdf_path)
                        documents.extend(pdf_documents)
                        logger.info(f"Successfully loaded PDF: {pdf_path}")
                    except Exception as e:
                        logger.error(f"Error loading PDF {pdf_path}: {str(e)}")
                        continue
                
                if not documents:
                    raise ValueError("No documents were successfully loaded")
                
                # Split documents into chunks
                logger.info('Splitting documents into chunks')
                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=200
                )
                texts = text_splitter.split_documents(documents)
                
                # Create and save vector store
                logger.info('Creating vector store')
                self.vector_store = FAISS.from_documents(texts, self.embeddings)
                self.vector_store.save_local(self.vector_store_path)
                logger.info("Created and saved new vector store")
        
        except Exception as e:
            logger.error(f"Error setting up vector store: {str(e)}")
            raise

    def generate_qa_pairs(self, num_pairs: int = 50) -> List[QuestionAnswer]:
        """Generate question-answer pairs using RAG and Claude.
        
        Args:
            num_pairs: Number of QA pairs to generate
            
        Returns:
            List of QuestionAnswer objects
        """
        qa_generator_prompt = """
        Generate a data science interview question and comprehensive answer based on the provided context.
        The question should be challenging but clear, and the answer should demonstrate mastery of the concept.
        
        Format:
        {
            "question": "Question text",
            "answer": "Detailed answer",
            "category": "One of: ML, Statistics, Programming, Math, Domain Knowledge",
            "difficulty": "One of: Easy, Medium, Hard"
        }
        """
        
        generated_pairs = []
        for _ in range(num_pairs):
            try:
                # Get random documents from vector store
                docs = self.vector_store.similarity_search(
                    "data science concept",
                    k=3
                )
                context = " ".join([doc.page_content for doc in docs])
                source = docs[0].metadata.get('source_pdf', 'unknown')
                
                # Generate QA pair using Claude
                message = self.client.messages.create(
                    model="claude-3-opus-20240229",
                    max_tokens=2000,
                    temperature=0.7,
                    messages=[{
                        "role": "user",
                        "content": qa_generator_prompt + "\n\nContext: " + context
                    }]
                )
                
                qa_data = json.loads(message.content[0].text)
                qa_pair = QuestionAnswer(**qa_data, source=source)
                generated_pairs.append(qa_pair)
                
                # Add delay to respect rate limits
                time.sleep(1)
                
            except Exception as e:
                logger.error(f"Error generating QA pair: {str(e)}")
                continue
        
        self.qa_pairs.extend(generated_pairs)
        return generated_pairs

    def score_question(self, qa_pair: QuestionAnswer) -> float:
        """Score the quality of a question-answer pair using Claude.
        
        Args:
            qa_pair: QuestionAnswer object to score
            
        Returns:
            Float score between 0 and 100
        """
        scoring_prompt = """
        Score the following data science interview question and answer on a scale of 0-100.
        Consider:
        1. Relevance (25 points): Does it test essential data science knowledge?
        2. Clarity (25 points): Is the question clear and unambiguous?
        3. Technical Depth (25 points): Does it require deep understanding?
        4. Practical Value (25 points): Is it applicable to real-world scenarios?

        Question: {question}
        Answer: {answer}
        Category: {category}
        Difficulty: {difficulty}

        Provide score and breakdown in JSON format:
        {{"total_score": float, "breakdown": {{"relevance": float, "clarity": float, "technical_depth": float, "practical_value": float}}, "feedback": string}}
        """
        
        try:
            message = self.client.messages.create(
                model="claude-3-opus-20240229",
                max_tokens=1000,
                temperature=0.3,
                messages=[{
                    "role": "user",
                    "content": scoring_prompt.format(**qa_pair.__dict__)
                }]
            )
            
            result = json.loads(message.content[0].text)
            logger.info(f'Scoring Question Result: {result}')
            qa_pair.score = result["total_score"]
            qa_pair.feedback = result["breakdown"]
            
            return result["total_score"]
            
        except Exception as e:
            logger.error(f"Error scoring question: {str(e)}")
            return 0.0

    def select_interview_question(self) -> Optional[QuestionAnswer]:
        """Select a random question from the top 36 highest-scoring questions."""
        if not self.qa_pairs:
            logger.warning("No questions available to select from")
            return None
            
        # Sort by score and get top 36
        top_questions = sorted(
            self.qa_pairs, 
            key=lambda x: x.score, 
            reverse=True
        )[:36]
        
        if not top_questions:
            return random.choice(self.qa_pairs)
            
        return random.choice(top_questions)

    def evaluate_answer(self, question: QuestionAnswer, candidate_answer: str) -> Tuple[float, Dict]:
        """Evaluate a candidate's answer against the model answer using Claude.
        
        Args:
            question: QuestionAnswer object containing the question and model answer
            candidate_answer: The candidate's response
            
        Returns:
            Tuple of (score, feedback_dict)
        """
        evaluation_prompt = """
        Evaluate the candidate's answer against the model answer for this data science interview question.
        Score from 1-10 and provide detailed feedback.

        Question: {question}
        Model Answer: {model_answer}
        Candidate Answer: {candidate_answer}

        Provide evaluation in JSON format:
        {{"score": float, "feedback": {{"strengths": [str], "areas_for_improvement": [str], "missing_key_points": [str]}}}}
        """
        
        try:
            message = self.client.messages.create(
                model="claude-3-opus-20240229",
                max_tokens=1000,
                temperature=0.3,
                messages=[{
                    "role": "user",
                    "content": evaluation_prompt.format(
                        question=question.question,
                        model_answer=question.answer,
                        candidate_answer=candidate_answer
                    )
                }]
            )
            
            result = json.loads(message.content[0].text)
            return result["score"], result["feedback"]
            
        except Exception as e:
            logger.error(f"Error evaluating answer: {str(e)}")
            return 0.0, {"error": str(e)}

    def save_qa_pairs(self, filepath: str = "qa_pairs.json"):
        """Save generated QA pairs to a JSON file."""
        try:
            with open(filepath, 'w') as f:
                json.dump([qa.__dict__ for qa in self.qa_pairs], f, indent=2)
            logger.info(f"Successfully saved QA pairs to {filepath}")
        except Exception as e:
            logger.error(f"Error saving QA pairs: {str(e)}")

    def load_qa_pairs(self, filepath: str = "qa_pairs.json"):
        """Load QA pairs from a JSON file."""
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                self.qa_pairs = [QuestionAnswer(**qa_data) for qa_data in data]
            logger.info(f"Successfully loaded QA pairs from {filepath}")
        except Exception as e:
            logger.error(f"Error loading QA pairs: {str(e)}")

def main():
    """Example usage of the interview system."""
    # Load environment variables
    anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
    if not anthropic_api_key:
        raise ValueError("ANTHROPIC_API_KEY environment variable not set")

    # Initialize system with multiple PDFs
    pdf_files = ["ds1.pdf", "ds2.pdf", "ds3.pdf"]  # Add your PDF files here
    interview_system = InterviewSystem(
        anthropic_api_key=anthropic_api_key,
        pdf_paths=pdf_files
    )

    try:
        # Generate QA pairs
        print("Generating question-answer pairs...")
        qa_pairs = interview_system.generate_qa_pairs(num_pairs=10)
        print(f"Generated {len(qa_pairs)} QA pairs")

        # Score questions
        print("\nScoring questions...")
        for qa_pair in qa_pairs:
            score = interview_system.score_question(qa_pair)
            print(f"Question scored: {score}")

        # Save QA pairs
        interview_system.save_qa_pairs()

        # Select and ask a question
        print("\nSelecting interview question...")
        selected_question = interview_system.select_interview_question()
        if selected_question:
            print("\nSelected Question:")
            print(f"Q: {selected_question.question}")
            print(f"Category: {selected_question.category}")
            print(f"Difficulty: {selected_question.difficulty}")
            
            # Get candidate's answer
            print("\nPlease provide your answer (press Enter twice to finish):")
            lines = []
            while True:
                line = input()
                if line == "":
                    break
                lines.append(line)
            candidate_answer = "\n".join(lines)
            
            # Evaluate answer
            print("\nEvaluating answer...")
            score, feedback = interview_system.evaluate_answer(
                selected_question, 
                candidate_answer
            )
            
            # Display results
            print(f"\nScore: {score}/10")
            print("\nFeedback:")
            print(json.dumps(feedback, indent=2))

    except Exception as e:
        logger.error(f"Error in interview process: {str(e)}")
        raise

if __name__ == "__main__":
    main()